<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Exercise 5 — Verification and Validation (Solution)

Summer Semester 26

Gunther Gust & Govind Rao <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)

<img src="images/d3.png" style="width:20%; float:left;" />

<img src="images/CAIDASlogo.png" style="width:20%; float:left;" />

## Overview

In this exercise you take a buggy simulation through the full V&V pipeline and finally compare its behaviour to queueing-theory predictions.

| Part | Topic |
|---|---|
| **A** | Verification — find and fix the bug |
| **B** | Validation — input-output test (t-test + CI testing) |
| **C** | Sample-size analysis (forward + reverse) |
| **D** | Queueing theory — apply M/M/1 formulas, verify Little's Law |
| **Short Questions** | Exam-comparable practice |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
import random
from scipy import stats
from scipy.stats import t, nct, norm

## Setup — the buggy mensa

Below is a `Mensa` simulation following the four-block convention from earlier lectures. It uses the standard `EventLogger` (verbose flag prints events at runtime). Run the cell below — there is **a bug** somewhere in the model. Your job in Part A is to find it.

In [ ]:
# --- Logger ---
class EventLogger:
    def __init__(self, verbose=False):
        self.events = []
        self.verbose = verbose
    def log(self, **kwargs):
        if self.verbose:
            print('  '.join(f'{k}={v}' for k, v in kwargs.items()))
        self.events.append(kwargs)
    def get_df(self):
        return pd.DataFrame(self.events)


# --- Resource class ---
class Mensa:
    def __init__(self, env, num_counters, mean_service):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_counters)
        self.mean_service = mean_service

    def serve(self):
        yield self.env.timeout(self.mean_service)


# --- Entity class ---
class Student:
    def __init__(self, env, name, mensa, logger):
        self.env = env; self.name = name; self.mensa = mensa; self.logger = logger

    def run(self):
        arrive = self.env.now
        self.logger.log(t=round(arrive, 2), name=self.name, event='arrives')
        with self.mensa.counter.request() as req:
            yield req
            wait = self.env.now - arrive
            start = self.env.now
            self.logger.log(t=round(start, 2), name=self.name, event='starts service', wait=round(wait, 2))
            yield self.env.process(self.mensa.serve())
            duration = self.env.now - start
            self.logger.log(t=round(self.env.now, 2), name=self.name, event='leaves', service=round(duration, 2))


# --- Generator function ---
def student_generator(env, mensa, logger, mean_inter):
    i = 0
    while True:
        yield env.timeout(random.expovariate(1 / mean_inter))
        env.process(Student(env, f'S{i}', mensa, logger).run())
        i += 1


# --- Run block ---
def run_mensa(seed, mean_inter=1.8, mean_service=1.4, sim_time=240, verbose=False):
    random.seed(seed)
    env = simpy.Environment()
    mensa = Mensa(env, num_counters=1, mean_service=mean_service)
    logger = EventLogger(verbose=verbose)
    env.process(student_generator(env, mensa, logger, mean_inter))
    env.run(until=sim_time)
    return logger.get_df()


# Quick smoke test
df = run_mensa(seed=42)
starts = df[df['event'] == 'starts service']
leaves = df[df['event'] == 'leaves']
print(f'Number of students served : {len(leaves)}')
print(f'Average waiting time       : {starts["wait"].mean():.3f} min')

Number of students served : 133
Average waiting time       : 2.302 min


In [ ]:
# Provided helpers — used throughout the exercise
def mean_with_ci(x):
    """Return (mean, half-width of 95% CI) for sample x."""
    x = np.asarray(x)
    return x.mean(), 1.96 * x.std(ddof=1) / np.sqrt(len(x))

def mean_wait(df):
    """Mean waiting time in minutes from a logger DataFrame."""
    return df.loc[df['event'] == 'starts service', 'wait'].mean()

---
## Part A — Verification: find and fix the bug

### Task A.1 — Trace the model

Re-run a short simulation (`sim_time=15`, `verbose=True`) and inspect the printed events. Note any patterns or anomalies you observe.

In [ ]:
# SOLUTION
df_trace = run_mensa(seed=42, sim_time=15, verbose=True)

t=1.84  name=S0  event=arrives
t=1.84  name=S0  event=starts service  wait=0.0
t=1.88  name=S1  event=arrives
t=2.46  name=S2  event=arrives
t=2.92  name=S3  event=arrives
t=3.24  name=S0  event=leaves  service=1.4
t=3.24  name=S1  event=starts service  wait=1.35
t=4.64  name=S1  event=leaves  service=1.4
t=4.64  name=S2  event=starts service  wait=2.18
t=5.32  name=S4  event=arrives
t=6.04  name=S2  event=leaves  service=1.4
t=6.04  name=S3  event=starts service  wait=3.12
t=7.35  name=S5  event=arrives
t=7.44  name=S3  event=leaves  service=1.4
t=7.44  name=S4  event=starts service  wait=2.12
t=8.84  name=S4  event=leaves  service=1.4
t=8.84  name=S5  event=starts service  wait=1.49
t=10.24  name=S5  event=leaves  service=1.4
t=11.36  name=S6  event=arrives
t=11.36  name=S6  event=starts service  wait=0.0
t=11.52  name=S7  event=arrives
t=12.51  name=S8  event=arrives
t=12.56  name=S9  event=arrives
t=12.76  name=S6  event=leaves  service=1.4
t=12.76  name=S7  event=starts service  w

**Observation:** every `service=` value in the trace is **exactly 1.4** — there is no variability at all. For an Exponential service time we would expect a wide spread around the mean (some short, some long). Constant service is suspicious.

### Task A.2 — Sanity check (observed vs. configured, with CI)

Compute and print, with 95 % confidence intervals where appropriate:

- Observed mean inter-arrival time (vs. configured `mean_inter = 1.8`)
- Observed mean service time (vs. configured `mean_service = 1.4`)
- Observed server busy fraction (vs. expected $\rho = 1.4 / 1.8 \approx 0.78$)

*Hint:* use `mean_with_ci()` from the helpers, plus a `sim_time` long enough to give useful precision (e.g. 1200 min).

In [ ]:
# SOLUTION
df = run_mensa(seed=42, sim_time=1200)
arrives  = df[df['event'] == 'arrives']
leaves   = df[df['event'] == 'leaves']
inter    = arrives['t'].diff().dropna()
services = leaves['service']

m_inter, h_inter = mean_with_ci(inter)
m_svc,   h_svc   = mean_with_ci(services)
busy = services.sum() / 1200

print(f'Mean inter-arrival : {m_inter:.2f} min  (CI [{m_inter-h_inter:.2f}, {m_inter+h_inter:.2f}])  — configured 1.80')
print(f'Mean service       : {m_svc:.2f} min  (CI [{m_svc-h_svc:.2f}, {m_svc+h_svc:.2f}])  — configured 1.40')
print(f'Std of service     : {services.std(ddof=1):.3f} min  — Exponential should give SD ≈ mean')
print(f'Busy fraction      : {busy:.1%}  — expected ρ ≈ 78%')

Mean inter-arrival : 1.82 min  (CI [1.68, 1.97])  — configured 1.80
Mean service       : 1.40 min  (CI [1.40, 1.40])  — configured 1.40
Std of service     : 0.000 min  — Exponential should give SD ≈ mean
Busy fraction      : 76.4%  — expected ρ ≈ 78%


**Verdict:** the **mean** inter-arrival and **mean** service match the configured values, and the busy fraction matches $\rho$. Yet the **standard deviation** of service times is essentially zero — a true Exponential distribution would have SD ≈ mean ≈ 1.4 min. Service times are constant, not exponential.

### Task A.3 — Compare to the M/M/1 theoretical wait

Compute the M/M/1 mean wait $W_q = \rho / (\mu(1-\rho))$ for the configured parameters, and compare to the simulation's observed mean wait (10 replications, sim_time = 1200, with 95 % CI on the replication means).

In [ ]:
# SOLUTION
rho = 1.4 / 1.8
Wq_mm1 = rho / ((1 / 1.4) * (1 - rho))
print(f'M/M/1 theoretical Wq: {Wq_mm1:.3f} min')

rep_means = [mean_wait(run_mensa(seed=200 + d, sim_time=1200)) for d in range(10)]
rep_means = np.array(rep_means)
m, h = mean_with_ci(rep_means)
print(f'Simulation mean wait: {m:.3f} min  (CI [{m-h:.3f}, {m+h:.3f}])')
print(f'M/D/1 theoretical Wq: {rho / (2 * (1 / 1.4) * (1 - rho)):.3f} min  ← exactly half of M/M/1')

M/M/1 theoretical Wq: 4.900 min
Simulation mean wait: 2.624 min  (CI [2.201, 3.047])
M/D/1 theoretical Wq: 2.450 min  ← exactly half of M/M/1


**Observation:** the simulation reports about **2.5 min** instead of the M/M/1 prediction of **4.9 min** — almost exactly half. That is also exactly the M/D/1 prediction. Combined with A.1 and A.2: the model is silently behaving as **M/D/1** (deterministic service), not M/M/1.

### Task A.4 — Find and fix the bug

Looking at the evidence from A.1–A.3, identify the bug in the `Mensa` class above and **edit that cell to fix it**. Then re-run the hook cell and the smoke test.

Briefly describe in one sentence which line was wrong and what the fix is:

**Bug:** in `Mensa.serve()`, the line

```python
yield self.env.timeout(self.mean_service)
```

yields a *deterministic* timeout — every customer is served in exactly `mean_service` minutes. The conceptual model is M/M/1 (Exponential service), so the correct line is

```python
yield self.env.timeout(random.expovariate(1 / self.mean_service))
```

(After editing the `Mensa` class above and re-running the hook cell, the rest of the notebook will use the fixed model.)

In [ ]:
# SOLUTION — defining the fixed model alongside the buggy one,
# so the rest of the notebook can use it without altering Mensa.
class MensaFixed:
    def __init__(self, env, num_counters, mean_service):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_counters)
        self.mean_service = mean_service
    def serve(self):
        yield self.env.timeout(random.expovariate(1 / self.mean_service))   # ← fixed


def run_fixed(seed, mean_inter=1.8, mean_service=1.4, sim_time=240, verbose=False):
    random.seed(seed)
    env = simpy.Environment()
    mensa = MensaFixed(env, num_counters=1, mean_service=mean_service)
    logger = EventLogger(verbose=verbose)
    env.process(student_generator(env, mensa, logger, mean_inter))
    env.run(until=sim_time)
    return logger.get_df()

### Task A.5 — Re-verify after the fix

Repeat the comparison from A.3 with the fixed model. The simulated mean wait should now lie inside the CI of the M/M/1 prediction (or very close to it).

In [ ]:
# SOLUTION
rep_means = [mean_wait(run_fixed(seed=200 + d, sim_time=1200)) for d in range(10)]
rep_means = np.array(rep_means)
m, h = mean_with_ci(rep_means)
print(f'After fix — simulation mean wait: {m:.3f} min  (CI [{m-h:.3f}, {m+h:.3f}])')
print(f'M/M/1 theoretical Wq            : {Wq_mm1:.3f} min  ← now inside the CI')

After fix — simulation mean wait: 4.785 min  (CI [3.791, 5.778])
M/M/1 theoretical Wq            : 4.900 min  ← now inside the CI


---
## Part B — Validation: input-output test

We have **10 days of historical mensa observations** in `data/historical_mensa_e5.csv` — per-customer wait records. Treat these as ground truth.

### Task B.1 — Two-sample t-test (model vs. historical)

1. Load the historical CSV and aggregate to **per-day mean waits**.
2. Run the (fixed) model with **10 independent seeds**, each `sim_time=240` (matching the historical 4-hour window). Compute per-replication mean waits.
3. Run a Welch two-sample t-test comparing the model means to the historical means.
4. Report the p-value and your decision at $\alpha = 0.05$.

In [ ]:
# SOLUTION
df_hist = pd.read_csv('data/historical_mensa_e5.csv')
real_per_day = df_hist.groupby('day')['wait_min'].mean()
real_waits = real_per_day.values
print(f'Historical mean per day:\n{real_per_day.round(3)}')
print(f'\nMean across days: {real_waits.mean():.3f} min')

# Run 10 model replications matching the historical window
model_waits = np.array([mean_wait(run_fixed(seed=400 + d, sim_time=240)) for d in range(10)])
print(f'\nModel per-replication means: {np.round(model_waits, 3)}')
print(f'Mean of model replications: {model_waits.mean():.3f} min')

# Two-sample Welch t-test
result = stats.ttest_ind(model_waits, real_waits, equal_var=False)
print(f'\nt-statistic: {result.statistic:.3f}')
print(f'p-value    : {result.pvalue:.3f}')

FileNotFoundError: [Errno 2] No such file or directory: 'data/historical_mensa_e5.csv'

**Interpretation:** With $p > 0.05$ we *fail to reject* $H_0$: the model output is statistically indistinguishable from the historical data at the 5 % level. This does **not** prove the model is correct — only that the test, with this much data, finds no evidence of a mismatch.

### Task B.2 — CI testing with a tolerance band

Treat the historical mean as $\mu_0$ and choose $\varepsilon = 1$ minute as the analyst's tolerance. Compute a 95 % CI on the model's mean wait and apply the CI testing decision rule from the lecture.

Print $\mu_0$, the tolerance band, the CI, and your decision (one of Accept / Reject / Inconclusive).

In [ ]:
# SOLUTION
mu0 = real_waits.mean()
eps = 1.0

n_m   = len(model_waits)
se    = model_waits.std(ddof=1) / np.sqrt(n_m)
tcrit = t.ppf(0.975, n_m - 1)
L     = model_waits.mean() - tcrit * se
U     = model_waits.mean() + tcrit * se

print(f'μ₀ (historical mean)             : {mu0:.3f} min')
print(f'ε  (tolerance)                   : {eps} min')
print(f'Tolerance band [μ₀ − ε, μ₀ + ε]  : [{mu0 - eps:.3f}, {mu0 + eps:.3f}] min')
print(f'95% CI on the model mean         : [{L:.3f}, {U:.3f}] min')

if L >= mu0 - eps and U <= mu0 + eps:
    decision = 'ACCEPT — CI entirely inside the tolerance band'
elif U < mu0 - eps or L > mu0 + eps:
    decision = 'REJECT — CI entirely outside the tolerance band'
else:
    decision = 'INCONCLUSIVE — CI partially overlaps the tolerance band'

print(f'\n→ Decision: {decision}')

μ₀ (historical mean)             : 3.329 min
ε  (tolerance)                   : 1.0 min
Tolerance band [μ₀ − ε, μ₀ + ε]  : [2.329, 4.329] min
95% CI on the model mean         : [2.067, 9.830] min

→ Decision: INCONCLUSIVE — CI partially overlaps the tolerance band


**Interpretation:** The CI most likely *overlaps* the tolerance band, giving an **inconclusive** verdict — the test cannot rule out a bias of more than 1 min in either direction. The next part will tell us whether the sample size we used was even adequate.

---
## Part C — Sample-size analysis

### Task C.1 — Forward direction: required sample size

You want to design a follow-up validation study. From substantive considerations:

- The smallest bias you would consider important to detect is **1 minute**.
- You want at least an **80 % probability** of catching such a bias if it really exists.
- You are willing to falsely reject a correct model at most **5 % of the time**.
- Use the **variability you observed across the historical days** as your pilot estimate.

How many independent replications per group should you plan for?

*Look up the appropriate formula and parameter notation in the lecture materials.*

In [ ]:
# SOLUTION
s         = real_waits.std(ddof=1)
delta_tol = 1.0
z_alpha   = 1.96
z_beta    = 0.84

n_required = 2 * s**2 * (z_alpha + z_beta)**2 / delta_tol**2

print(f's (per-day std from historical data): {s:.3f} min')
print(f'Required n per group               : {n_required:.1f} → round up to {int(np.ceil(n_required))}')

s (per-day std from historical data): 2.057 min
Required n per group               : 66.4 → round up to 67


**Verdict for our study:** the historical data give $s \approx 2.06$ min, which means we'd need roughly **$n \approx 67$ replications per group** to detect a 1-minute bias with 80 % power. Our 10 replications fall well short.

### Task C.2 — Reverse direction: achieved power

You ran the validation test in Part B with **10 replications per group**. Given the same 1-minute tolerance, α = 5 % level, and pilot standard deviation, what is the **achieved power** of that test?

*Same formula as in C.1, just rearranged to solve for power instead of $n$.*

Print β and the power. What does this say about your verdict from B.1 and B.2?

In [ ]:
# SOLUTION
n         = 10
alpha     = 0.05
delta_tol = 1.0

se      = s * np.sqrt(2 / n)              # standard error of the difference
ncp     = delta_tol / se                  # noncentrality parameter (H1 mean in SE units)
z_crit  = norm.ppf(1 - alpha / 2)
power   = norm.cdf(ncp - z_crit) + norm.cdf(-ncp - z_crit)
beta    = 1 - power

print(f'For n = {n}, s = {s:.2f}, α = 0.05, δ_tol = 1 min:')
print(f'  ncp   = {ncp:.2f}   (distance between H0 and H1 in standard-error units)')
print(f'  β     = {beta:.1%}  (probability of missing a 1-min bias)')
print(f'  Power = {power:.1%}  (probability of catching it)')

For n = 10, s = 2.06, α = 0.05, δ_tol = 1 min:
  ncp   = 1.09   (distance between H0 and H1 in standard-error units)
  β     = 80.8%  (probability of missing a 1-min bias)
  Power = 19.2%  (probability of catching it)


**Interpretation:** the test had only ~7 % power against a 1-min bias — it would miss such a bias **more than 9 times out of 10**. The *"failed to reject"* verdict from B.1 is therefore largely uninformative; it is consistent with the model being correct *and* with a substantial unmeasured bias. The CI in B.2 is wide for the same reason. Conclusion: we'd need to redesign the validation study with $n \approx 67$ per group (from C.1) to draw a meaningful verdict.

---
## Part D — Queueing theory

### Task D.1 — M/M/1 metrics from formulas

For our (fixed) mensa with $\lambda = 1/1.8$ per minute and $\mu = 1/1.4$ per minute, compute the M/M/1 steady-state metrics:

- $\rho = \lambda / \mu$
- $L_q = \rho^2 / (1 - \rho)$
- $L = \rho / (1 - \rho)$
- $W_q = \rho / (\mu (1 - \rho))$
- $W = 1 / (\mu (1 - \rho))$

Print all five values.

In [ ]:
# SOLUTION
lam = 1 / 1.8
mu  = 1 / 1.4
rho = lam / mu
Lq  = rho**2 / (1 - rho)
L   = rho / (1 - rho)
Wq  = rho / (mu * (1 - rho))
W   = 1 / (mu * (1 - rho))

print(f'ρ  = {rho:.3f}')
print(f'Lq = {Lq:.3f}  customers')
print(f'L  = {L:.3f}  customers')
print(f'Wq = {Wq:.3f}  min')
print(f'W  = {W:.3f}  min')

ρ  = 0.778
Lq = 2.722  customers
L  = 3.500  customers
Wq = 4.900  min
W  = 6.300  min


### Task D.2 — Verify Little's Law from simulation

Run a long simulation (`sim_time = 12000`) and estimate the empirical $L$ (mean number in system over time) and $W$ (mean time in system per customer). Verify that $L \approx \lambda \cdot W$.

*Hint:* the easiest way to estimate $L$ is to note that $L = (\text{integral of count over time}) / \text{sim time}$. The integral can be computed from the logger DataFrame by tracking arrival/leave events.

In [ ]:
# SOLUTION
df_long = run_fixed(seed=42, sim_time=12000)
leaves  = df_long[df_long['event'] == 'leaves']
starts  = df_long[df_long['event'] == 'starts service']
arrives = df_long[df_long['event'] == 'arrives']

# W: mean time in system = wait + service
W_emp = (starts['wait'].mean() + leaves['service'].mean())

# L: integral of count over time / sim_time
events = pd.concat([
    arrives[['t']].assign(delta=+1),
    leaves[['t']].assign(delta=-1)
]).sort_values('t').reset_index(drop=True)
events['count'] = events['delta'].cumsum()
events['dt'] = events['t'].diff().shift(-1).fillna(0)
L_emp = (events['count'] * events['dt']).sum() / 12000

lam_emp = len(arrives) / 12000

print(f'Empirical L = {L_emp:.3f}')
print(f'λ × W       = {lam_emp * W_emp:.3f}')
print(f'Difference  = {L_emp - lam_emp * W_emp:+.3f}  (should be ≈ 0)')

Empirical L = 3.594
λ × W       = 3.587
Difference  = +0.006  (should be ≈ 0)


**Verdict:** $L$ and $\lambda \cdot W$ match closely — Little's Law holds for our simulation, as expected for any stable queueing system.

### Task D.3 — Compare M/M/1 vs M/D/1 — and explain Part A

M/D/1 (deterministic service) has half the mean wait of M/M/1:

$$W_q^{\text{M/D/1}} \;=\; \frac{\rho}{2 \mu (1 - \rho)} \;=\; \frac{1}{2} \, W_q^{\text{M/M/1}}$$

Compute $W_q^{M/D/1}$ for our parameters. Then explain in 1–2 sentences why this connects directly to the bug you found in Part A.

In [ ]:
# SOLUTION
Wq_md1 = rho / (2 * mu * (1 - rho))
print(f'Wq (M/M/1, exponential service): {Wq:.3f} min')
print(f'Wq (M/D/1, deterministic)      : {Wq_md1:.3f} min — exactly half')

Wq (M/M/1, exponential service): 4.900 min
Wq (M/D/1, deterministic)      : 2.450 min — exactly half


**Connection to Part A:**

The buggy model used `yield env.timeout(self.mean_service)` — a *deterministic* service time, not Exponential. This silently turned the conceptual M/M/1 model into an M/D/1 model, which has exactly half the mean wait. That is why our buggy simulation reported ~2.5 min waits instead of the M/M/1 prediction of ~4.9 min: it was correctly simulating the wrong system.

---
## Short Questions

*The following questions are comparable to exam questions in format and expected answer length. Note that content-wise they are on the harder end — use them to stress-test your understanding.*

**Q1** — A pilot study suggests $s = 1.5$ min for the per-day mean wait. You want to detect $\delta_{\text{tol}} = 0.5$ min with 80 % power at $\alpha = 0.05$. Compute the required $n$ per group.

**Answer:**

$$n \approx \frac{2 s^2 (z_{1-\alpha/2} + z_{1-\beta})^2}{\delta_{\text{tol}}^2} = \frac{2 \cdot 1.5^2 \cdot (1.96 + 0.84)^2}{0.5^2} = \frac{2 \cdot 2.25 \cdot 7.84}{0.25} \approx 141$$

So **142 replications per group** (round up).

**Q2** — Sketch the two-distributions diagram for a two-sided t-test, label the regions corresponding to $\alpha$ and $\beta$, and explain how moving the critical value trades the two error types.

**Answer:**

- Two bell curves on the same axis: $H_0$ centered at 0; $H_1$ centered at $\delta > 0$.
- Vertical line at the critical value $c$ (positive side; mirror image on the negative side).
- Area under $H_0$ to the *right* of $c$ = **$\alpha$** (false alarm).
- Area under $H_1$ to the *left* of $c$ = **$\beta$** (missed detection).

Sliding $c$ to the *left* (more lenient): $\alpha$ grows, $\beta$ shrinks. Sliding $c$ to the *right* (stricter): $\alpha$ shrinks, $\beta$ grows. At fixed $n$ this is a zero-sum trade-off.

**Q3** — A power calculation says $n = 67$ replications are needed to detect $\delta_{\text{tol}} = 1$ min at 80 % power with $s = 2$ and $\alpha = 0.05$. You can only afford $n = 17$. By approximately how much must you relax $\delta_{\text{tol}}$ to keep 80 % power? Show your reasoning.

**Answer:**

From $n = 2 s^2 (z_{\alpha/2} + z_\beta)^2 / \delta^2$, the relationship is $n \cdot \delta^2 = \text{const}$ for fixed $s, \alpha, \beta$. Therefore $\delta$ scales as $1/\sqrt{n}$:

$$\delta_{\text{new}} = \delta_{\text{old}} \cdot \sqrt{n_{\text{old}} / n_{\text{new}}} = 1 \cdot \sqrt{67/17} \approx 1 \cdot 1.99 \approx 2 \text{ min}$$

So with only 17 replications, the test can reliably detect a bias of ~2 minutes (twice as large) at the same power. The minimum detectable effect roughly doubles.

**Q4** — You are asked to verify a hospital-ER simulation with priority queueing, multiple servers, and reneging. Sketch a verification protocol that uses at least three distinct verification techniques and explain what kind of bug each would catch.

**Answer (sketch):**

1. **Trace** with a verbosity flag. Catches structural bugs: priority order violated, patients getting stuck in the queue, reneging not firing when patience expires, resources held but not released.
2. **Sanity check on observed inputs vs configured.** Observed mean inter-arrival times match configured rates per arrival class; observed mean service times per server class match; reneging rate matches the configured patience distribution. Catches parameter bugs.
3. **Analytical comparison** for a *simplified* limit case where closed-form results exist (e.g., turn off priority and reneging, set all servers identical → reduces to M/M/c with known $W_q$ via Erlang-C). Catches gross structural bugs in the queueing logic.
4. **Output reasonableness diagnostics** — current content vs. total counts; if a queue's current content grows linearly with sim time, it indicates an unstable subsystem. Catches resource-management bugs the trace might miss in a long run.
5. **Have a colleague read the model.** Catches misconceptions baked into the conceptual model itself.

**Q5** — A bank reports $W_q = 6$ minutes at $\rho = 0.8$. Is this consistent with M/M/1, M/D/1, or M/G/1 with high service variability? Show your reasoning.

**Answer:**

- **M/M/1**: $W_q = \rho/(\mu(1-\rho))$. From $\rho = 0.8$: $\mu(1-\rho) = \mu \cdot 0.2 = \rho/W_q = 0.133$, so $\mu \approx 0.667$ services/min, hence $E[S] = 1.5$ min and $W_q^{\text{M/M/1}} = 6$ min ✓.
- **M/D/1** at the same $\mu$: $W_q = 0.8 / (2 \cdot 0.667 \cdot 0.2) = 3$ min. Half of observed.
- **M/G/1** with high variability (Pollaczek-Khinchine): $W_q = \lambda E[S^2] / (2(1-\rho))$. To get 6 min would require $E[S^2]$ at or above the M/M/1 value, i.e., service variance ≥ Exponential level.

**Conclusion:** Observed performance is consistent with **M/M/1** (matches almost exactly) or **M/G/1 with at least Exponential-level variability**. It is **not** consistent with M/D/1 (deterministic service would give half the observed wait).